In [ ]:
pip install pandas numpy scikit-learn transformers datasets torch

In [ ]:
!pip install -U accelerate
!pip install -U transformers

In [14]:
!nvidia-smi

Wed Apr 22 15:37:15 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.274.02             Driver Version: 535.274.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   71C    P0              30W /  70W |  14875MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
import pandas as pd
import numpy as np
import re
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# ==========================================
# 1단계: 데이터 로드 및 전처리 (Data Cleaning)
# ==========================================
print("▶ [1/5] 데이터 로드 및 전처리를 시작합니다...")

# 데이터 불러오기
train_combined_path = "normal_conversations/train_combined.csv"
df = pd.read_csv(train_combined_path)

# 공백 제거 및 라벨 매핑 (모델은 숫자로 된 라벨만 이해합니다)
df['class'] = df['class'].str.strip()
label_mapping = {
    '협박 대화': 0,
    '갈취 대화': 1,
    '직장 내 괴롭힘 대화': 2,
    '기타 괴롭힘 대화': 3,
    '일반 대화': 4
}
df['label'] = df['class'].map(label_mapping)

# 결측치 확인 및 제거 (만약 매핑이 안 된 데이터가 있다면 제거)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# 텍스트 정제 함수: 한글, 영문, 숫자, 기본 문장부호만 남기고 특수문자 제거
def standard_preprocess(text):
    if not isinstance(text, str):
        return ""

    # 1. 줄바꿈, 탭 등 공백 문자 정규화
    text = re.sub(r'\s+', ' ', text)

    # 2. URL 및 이메일 제거 (선택 사항)
    text = re.sub(r'http\S+|www\S+|mailto:\S+', '', text)

    # 3. 특수문자 정제 (한글, 숫자, 기본 문장부호만 남기기)
    # KLUE 모델은 문장부호를 이해하므로 .?!,는 남겨두는 것이 좋습니다.
    text = re.sub(r'[^가-힣0-9a-zA-Zㄱ-ㅎㅏ-ㅣ\s.?!,]', '', text)

    return text.strip()
    
df['clean_conversation'] = df['conversation'].apply(standard_preprocess)

# ==========================================
# 2단계: 데이터 세트 분할 (Train/Validation Split)
# ==========================================
print("▶ [2/5] 데이터를 8:2로 분할합니다...")

# 8:2 비율로 나누기 (stratify=df['label']을 통해 각 클래스 비율을 균등하게 유지)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

print(f"  - 훈련 데이터(Train): {len(train_df)}개")
print(f"  - 검증 데이터(Validation): {len(val_df)}개")

# Hugging Face Dataset 형식으로 변환
train_dataset = Dataset.from_pandas(train_df[['clean_conversation', 'label']])
val_dataset = Dataset.from_pandas(val_df[['clean_conversation', 'label']])

# ==========================================
# 3단계: 토큰화 (Tokenization)
# ==========================================
print("▶ [3/5] KLUE-RoBERTa 토크나이저를 적용합니다...")

MODEL_NAME = "klue/roberta-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    # 최대 길이를 600로 제한하고, 모자란 부분은 패딩(padding) 처리
    return tokenizer(examples["clean_conversation"], padding="max_length", truncation=True, max_length=512)

# 데이터셋 전체에 토큰화 일괄 적용
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# 모델 학습에 불필요한 텍스트 컬럼 제거
tokenized_train = tokenized_train.remove_columns(["clean_conversation", "__index_level_0__"])
tokenized_val = tokenized_val.remove_columns(["clean_conversation", "__index_level_0__"])

# ==========================================
# 4단계: 모델 설정 및 학습 (Modeling & Training)
# ==========================================
print("▶ [4/5] 모델 로드 및 학습을 시작합니다. (시간이 다소 소요됩니다)")

# 분류용 모델 로드 (num_labels=5)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)

# 평가지표(Metrics) 계산 함수 정의: Accuracy와 F1-Score(Macro) 반환
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro') # 범죄 분류에서는 클래스 불균형을 고려해 macro 사용
    
    return {
        'accuracy': acc,
        'f1_macro': f1
    }

# 학습 설정 (Hyperparameters)
training_args = TrainingArguments(
    output_dir='./results',          # 모델 예측/체크포인트 저장 폴더
    num_train_epochs=5,              # 전체 데이터 반복 학습 횟수
    per_device_train_batch_size=16,  # 1회 학습 데이터 수 (GPU 메모리에 따라 조절: 8 or 16)
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    fp16 = True,
    learning_rate=1e-5,              # 학습률
    weight_decay=0.01,
    eval_strategy="epoch",     # 매 에포크마다 검증 데이터로 성능 평가
    save_strategy="epoch",
    logging_steps=50,           # 로그를 더 자주 찍어 상태 확인
    report_to="none",           # 외부 툴 연동 제외 (깔끔한 로그용)
    load_best_model_at_end=True,     # 학습 종료 후 가장 성능이 좋았던 모델 로드
    metric_for_best_model="f1_macro" # 최고 성능 기준을 f1_score로 설정
)

# Trainer 객체 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# 본격적인 학습 시작!
trainer.train()

for callback in trainer.callback_handler.callbacks:
    if "NotebookProgressCallback" in str(type(callback)):
        trainer.remove_callback(callback)
        break
        
# ==========================================
# 5단계: 최종 평가 (Evaluation)
# ==========================================
print("▶ [5/5] 최종 모델 성능 평가를 진행합니다...")

eval_results = trainer.evaluate()

print("-" * 50)
print("🎯 [최종 검증 세트(Validation Set) 평가 결과]")
print(f" - Loss (손실): {eval_results['eval_loss']:.4f}")
print(f" - Accuracy (정확도): {eval_results['eval_accuracy'] * 100:.2f}%")
print(f" - F1 Score (Macro): {eval_results['eval_f1_macro']:.4f}")
print("-" * 50)


from sklearn.metrics import classification_report
import numpy as np

# 1. 검증 데이터셋에 대해 예측 수행
print("▶ 검증 데이터셋 예측 중...")
# 토큰화된 검증 데이터셋(tokenized_val)을 사용해야 합니다.
output = trainer.predict(tokenized_val)
preds = np.argmax(output.predictions, axis=-1)

# 2. 결과 리포트 출력
# label_mapping의 키값(한글 이름)들을 순서대로 가져옵니다.
target_names = ['협박 대화', '갈취 대화', '직장 내 괴롭힘 대화', '기타 괴롭힘 대화', '일반 대화']

print("\n" + "="*60)
print("🎯 [KLUE-RoBERTa 상세 성능 리포트]")
print("="*60)
# val_df['label']은 실제 정답, preds는 모델의 예측값입니다.
print(classification_report(tokenized_val['label'], preds, target_names=target_names))
print("="*60)

# 학습된 모델 저장 (선택 사항)
# trainer.save_model("./best_klue_roberta_model")

▶ [1/5] 데이터 로드 및 전처리를 시작합니다...
▶ [2/5] 데이터를 8:2로 분할합니다...
  - 훈련 데이터(Train): 3960개
  - 검증 데이터(Validation): 990개
▶ [3/5] KLUE-RoBERTa 토크나이저를 적용합니다...


Map:   0%|          | 0/3960 [00:00<?, ? examples/s]

Map:   0%|          | 0/990 [00:00<?, ? examples/s]

▶ [4/5] 모델 로드 및 학습을 시작합니다. (시간이 다소 소요됩니다)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

▶ [5/5] 최종 모델 성능 평가를 진행합니다...
--------------------------------------------------
🎯 [최종 검증 세트(Validation Set) 평가 결과]
 - Loss (손실): 0.2014
 - Accuracy (정확도): 93.84%
 - F1 Score (Macro): 0.9382
--------------------------------------------------
▶ 검증 데이터셋 예측 중...

🎯 [KLUE-RoBERTa 상세 성능 리포트]
              precision    recall  f1-score   support

       협박 대화       0.91      0.89      0.90       179
       갈취 대화       0.89      0.95      0.92       196
 직장 내 괴롭힘 대화       0.96      0.96      0.96       196
   기타 괴롭힘 대화       0.92      0.89      0.90       219
       일반 대화       1.00      1.00      1.00       200

    accuracy                           0.94       990
   macro avg       0.94      0.94      0.94       990
weighted avg       0.94      0.94      0.94       990

